# 📊 Evaluasi Sistem Rekomendasi Wisata Danau Toba
## Perbandingan Performa RAG vs CAG

**Model LLM:** Gemini 2.0 Flash (Google API)

### Metrik yang Diukur:
| Kategori | Metrik | Deskripsi |
|----------|--------|-----------|
| **Efficiency** | Response Time | Waktu total dari query hingga response |
| **Efficiency** | Cache Hit Rate (CHR) | Persentase query yang dilayani dari cache |
| **Retrieval** | RAG Recall | Keyword relevan yang ditemukan di retrieved docs |
| **Retrieval** | EIR | Effective Information Rate - info context yang digunakan |
| **Generation** | BERTScore F1 | Semantic similarity dengan ground truth |
| **Generation** | Completeness | Coverage keyword dalam response |
| **Generation** | Hallucination Rate | Informasi yang tidak ada di context |

## 1️⃣ Setup Environment
Install dependencies dan import libraries yang diperlukan.

In [ ]:
# Install Dependencies
%pip install -q python-dotenv sentence-transformers langchain langchain-community faiss-cpu bert-score pandas matplotlib seaborn

In [ ]:
# Setup & Imports
import sys, os, json, time
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from datetime import datetime
from bert_score import score as bert_score
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv

sys.path.append(os.path.abspath('../src'))
load_dotenv()
sns.set_style('whitegrid')
print("✅ Setup complete")

## 2️⃣ Load Model & Encoder
- **LLM:** Gemini 2.0 Flash via Google API (tidak memerlukan GPU lokal)
- **Encoder:** sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 untuk embedding dokumen (multilingual, 50+ bahasa termasuk Indonesia)

In [ ]:
# Load Gemini 2.0 Flash & Encoder
from model import GeminiChatModel
gemini = GeminiChatModel(model_name="gemini-2.0-flash")
encoder = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("✅ Gemini 2.0 Flash & Encoder loaded")

## 3️⃣ Build Knowledge Base
Load dokumen PDF → Split chunks → Buat FAISS vector database untuk retrieval.

In [ ]:
# Load Documents & Build Vector DB
tourism_dir = os.path.abspath('../database/vectordatabase')
pdf_files   = [os.path.join(tourism_dir, f)
               for f in os.listdir(tourism_dir) if f.endswith('.pdf')] \
              if os.path.exists(tourism_dir) else []

pages = [p for pdf in pdf_files for p in PyPDFLoader(pdf).load()]
# chunk_size=768, overlap=100 — sinkron dengan cag_system.py (lebih baik untuk data wisata berbahasa Indonesia)
docs  = RecursiveCharacterTextSplitter(chunk_size=768, chunk_overlap=100).split_documents(pages) if pages else []
faiss_db = FAISS.from_documents(docs, encoder) if docs else None

if pdf_files:
    print(f"✅ {len(pdf_files)} PDFs → {len(docs)} chunks  ({tourism_dir})")
else:
    print(f"⚠️  Tidak ada PDF di {tourism_dir}")
    print("   Letakkan file PDF wisata ke folder database/vectordatabase/")

## 4️⃣ Setup CAG System
Inisialisasi **Cache-Augmented Generation** untuk caching respons dan mempercepat query berulang.

In [ ]:
# Setup CAG System
try:
    from cag_system import CAGSystem
    cag = CAGSystem(gemini, encoder)
    if pdf_files: cag.load_documents(pdf_files, use_summaries=False)
    cag_ok = True
    print("✅ CAG ready")
except Exception as e:
    cag_ok = False
    print(f"⚠️ CAG not available: {e}")

## 5️⃣ Test Dataset
5 query uji tentang wisata Danau Toba dengan **ground truth** dan **keywords** untuk evaluasi.

In [ ]:
# === Dataset Evaluasi ===
# Prioritas 1 : load dari FAQ (ground truth resmi, entries yang punya 'answer')
# Prioritas 2 : fallback ke 5 sample queries jika FAQ belum punya 'answer'

faq_path = os.path.abspath('../database/FAQ/faq_tourism.json')
dataset  = []

if os.path.exists(faq_path):
    with open(faq_path, 'r', encoding='utf-8') as _f:
        _faq_data = json.load(_f)

    # Gunakan hanya FAQ yang sudah memiliki field 'answer' sebagai ground truth
    dataset = [
        {
            "q":  faq["question"],
            "gt": faq["answer"],
            "kw": faq.get("keywords", []),
        }
        for faq in _faq_data
        if faq.get("answer", "").strip()
    ]

    if dataset:
        print(f"✅ Loaded {len(dataset)} FAQ entries (dengan 'answer') sebagai ground truth")
        print(f"   Contoh: Q={dataset[0]['q'][:50]}")
        print(f"           GT={dataset[0]['gt'][:60]}")
    else:
        print("⚠️  FAQ ditemukan tapi belum ada field 'answer'.")
        print("   Setelah mengirimkan 500 FAQ+jawaban, re-run cell ini.")

# Fallback: 5 sample queries jika FAQ belum ada jawaban
if not dataset:
    dataset = [
        {"q": "Rekomendasi wisata Danau Toba untuk keluarga?",
         "gt": "Pulau Samosir, Museum Huta Bolon, Pantai Parapat, Desa Tomok.",
         "kw": ["Samosir", "Parapat", "wisata", "Batak"]},
        {"q": "Biaya homestay di Danau Toba?",
         "gt": "Rp 150.000 - Rp 500.000 per malam.",
         "kw": ["homestay", "biaya", "Rp"]},
        {"q": "Makanan khas Batak apa saja?",
         "gt": "Saksang, Arsik, Naniura, Dali ni Horbo.",
         "kw": ["Saksang", "Arsik", "Batak"]},
        {"q": "Cara ke Pulau Samosir dari Parapat?",
         "gt": "Ferry setiap 30 menit, Rp 10.000.",
         "kw": ["ferry", "Parapat", "Samosir"]},
        {"q": "Waktu terbaik ke Danau Toba?",
         "gt": "Mei-September (musim kemarau).",
         "kw": ["waktu", "musim", "kemarau"]},
    ]
    print(f"ℹ️  Menggunakan {len(dataset)} sample queries sebagai fallback")

print(f"\n📊 Total dataset: {len(dataset)} queries")

## 6️⃣ Inference Functions
- **RAG:** Retrieve → Generate (tanpa cache)
- **CAG:** Check cache → Retrieve → Generate → Cache response

In [ ]:
# === Inference Functions ===
#
# RAG  : use_cache=False  → selalu retrieve dari vector DB + generate (tidak pakai cache)
# CAG  : use_cache=True   → cek cache dulu; jika hit → langsung return,
#                           jika miss → retrieve + generate + simpan ke cache
#
# Kedua fungsi menggunakan cag.get_response() yang sama, perbedaannya hanya pada
# flag use_cache. Ini memastikan evaluasi RAG vs CAG apples-to-apples.

def rag_infer(q: str, k: int = 5) -> dict:
    """RAG pure: selalu retrieve + generate, tanpa cache."""
    if not cag_ok:
        return {'resp': '', 'ctx': '', 'time': 0, 'cache': False}
    t0 = time.time()
    r  = cag.get_response(q, k=k, use_cache=False)
    return {
        'resp':  r.get('response', ''),
        'ctx':   r.get('context',  ''),
        'time':  time.time() - t0,
        'cache': False,
    }


def cag_infer(q: str, k: int = 5) -> dict:
    """CAG: cache-first, lalu RAG jika miss."""
    if not cag_ok:
        return {'resp': '', 'ctx': '', 'time': 0, 'cache': False}
    t0 = time.time()
    r  = cag.get_response(q, k=k, use_cache=True)
    return {
        'resp':  r.get('response',   ''),
        'ctx':   r.get('context',    ''),
        'time':  time.time() - t0,
        'cache': r.get('cache_used', False),
    }


print("✅ Inference ready")
print("   RAG : use_cache=False  (pure retrieval+generate)")
print("   CAG : use_cache=True   (cache-first → RAG fallback)")

## 7️⃣ Evaluation Metrics
| Metrik | Formula |
|--------|---------|
| **BERTScore F1** | Semantic similarity response vs ground truth |
| **Completeness** | % keywords yang muncul di response |
| **Hallucination** | % kalimat response tanpa grounding di context |
| **RAG Recall** | % keywords ditemukan di retrieved docs |
| **EIR** | % context words yang digunakan di response |

In [ ]:
# === Metrics Functions ===

def bertscore_f1(preds: list, refs: list) -> float:
    """Semantic similarity antara response (preds) dan ground truth (refs)."""
    P, R, F = bert_score(preds, refs, lang='id', verbose=False)
    return F.mean().item()

def completeness(resp: str, kw: list) -> float:
    """Persentase keywords ground-truth yang muncul di response."""
    if not kw:
        return 0.0
    return sum(k.lower() in resp.lower() for k in kw) / len(kw)

def hallucination(resp: str, ctx: str) -> float:
    """
    Estimasi hallucination: proporsi kalimat di response yang tidak memiliki
    grounding (tidak ada kata panjang dari kalimat itu yang muncul di context).
    """
    if not ctx:
        return 0.0
    sents = [s.strip() for s in resp.split('.') if len(s.strip()) > 20]
    if not sents:
        return 0.0
    def _grounded(sent: str) -> bool:
        return any(w in ctx.lower() for w in sent.lower().split() if len(w) > 4)
    return sum(not _grounded(s) for s in sents) / len(sents)

def rag_recall(ctx: str, kw: list) -> float:
    """
    Persentase keywords yang ditemukan di context yang di-retrieve.
    Menggunakan ctx (string) bukan list dokumen.
    """
    if not kw or not ctx:
        return 0.0
    ctx_lower = ctx.lower()
    return sum(k.lower() in ctx_lower for k in kw) / len(kw)

def eir(ctx: str, resp: str) -> float:
    """
    Effective Information Rate: proporsi kata panjang dari context
    yang benar-benar digunakan di response.
    """
    if not ctx or not resp:
        return 0.0
    cw = {w.lower() for w in ctx.split()  if len(w) > 4}
    rw = {w.lower() for w in resp.split() if len(w) > 4}
    return len(cw & rw) / len(cw) if cw else 0.0

print("✅ Metrics ready")
print("   bertscore_f1 | completeness | hallucination | rag_recall | eir")

## 8️⃣ Run Evaluation
Eksekusi RAG dan CAG untuk setiap query, dengan rate limiting 2 detik per query.

In [ ]:
# 🚀 RUN EVALUATION
print("=" * 60)
print("🚀 EVALUATING RAG vs CAG")
print(f"   Dataset : {len(dataset)} queries")
print(f"   Interval: 3s/query (rate limit Gemini)")
print("=" * 60)

rag_res, cag_res = [], []

for i, d in enumerate(dataset, 1):
    print(f"\n[{i}/{len(dataset)}] {d['q'][:55]}…")

    # ── RAG (no cache) ──
    r = rag_infer(d['q'])
    r.update({'gt': d['gt'], 'kw': d['kw']})
    rag_res.append(r)
    print(f"  RAG : {r['time']:.2f}s  | ctx={len(r['ctx'])} chars")

    # ── CAG (cache-first) ──
    if cag_ok:
        c = cag_infer(d['q'])
        c.update({'gt': d['gt'], 'kw': d['kw']})
        cag_res.append(c)
        hit_label = "📦 CACHE HIT" if c['cache'] else "🔍 RAG"
        print(f"  CAG : {c['time']:.2f}s  | {hit_label}")

    time.sleep(3)   # hindari rate-limit Gemini

print(f"\n{'='*60}")
print(f"✅ Done!  RAG={len(rag_res)}  CAG={len(cag_res)}")

## 9️⃣ Calculate & Display Results
Hitung semua metrik dan tampilkan perbandingan RAG vs CAG dalam tabel.

In [ ]:
# 📊 CALCULATE METRICS
def calc_metrics(res: list) -> dict:
    m = {'time': [], 'bert': [], 'comp': [], 'hall': [], 'recall': [], 'eir': [], 'cache': []}
    for r in res:
        m['time'].append(r['time'])
        m['cache'].append(1 if r.get('cache') else 0)

        # BERTScore: response vs ground truth
        m['bert'].append(bertscore_f1([r['resp']], [r['gt']]) if r['resp'] else 0.0)

        # Completeness: keywords gt ditemukan di response
        m['comp'].append(completeness(r['resp'], r['kw']))

        # Hallucination: kalimat response tanpa grounding di context
        m['hall'].append(hallucination(r['resp'], r['ctx']))

        # RAG Recall: keywords gt ditemukan di context yang di-retrieve (pakai 'ctx')
        m['recall'].append(rag_recall(r['ctx'], r['kw']))

        # EIR: proporsi kata context yang terpakai di response
        m['eir'].append(eir(r['ctx'], r['resp']))

    return {k: float(np.mean(v)) if v else 0.0 for k, v in m.items()}

rag_m = calc_metrics(rag_res)
cag_m = calc_metrics(cag_res) if cag_res else None
print("✅ Metrics calculated")
print(f"   RAG avg response time : {rag_m['time']:.3f}s")
if cag_m:
    print(f"   CAG avg response time : {cag_m['time']:.3f}s")
    cache_rate = cag_m['cache'] * 100
    print(f"   CAG cache hit rate    : {cache_rate:.0f}%")

In [ ]:
# 📋 RESULTS TABLE
print("\n" + "="*60 + "\n📊 EVALUATION RESULTS\n" + "="*60)
metrics = ['Response Time (s)', 'Cache Hit Rate', 'BERTScore F1', 'Completeness', 'Hallucination', 'RAG Recall', 'EIR']
rag_v = [f"{rag_m['time']:.3f}", f"{rag_m['cache']*100:.0f}%", f"{rag_m['bert']:.4f}", f"{rag_m['comp']:.4f}", f"{rag_m['hall']:.4f}", f"{rag_m['recall']:.4f}", f"{rag_m['eir']:.4f}"]
data = {'Metric': metrics, 'RAG': rag_v}

if cag_m:
    data['CAG'] = [f"{cag_m['time']:.3f}", f"{cag_m['cache']*100:.0f}%", f"{cag_m['bert']:.4f}", f"{cag_m['comp']:.4f}", f"{cag_m['hall']:.4f}", f"{cag_m['recall']:.4f}", f"{cag_m['eir']:.4f}"]
    speedup = rag_m['time'] / cag_m['time'] if cag_m['time'] > 0 else 0

df = pd.DataFrame(data)
print(df.to_string(index=False))
if cag_m: print(f"\n🚀 CAG Speedup: {speedup:.2f}x")

## 🔟 Visualization
Bar charts untuk perbandingan visual: Response Time, BERTScore, dan Quality Metrics.

In [ ]:
# 📈 VISUALIZATION
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sys_names = ['RAG'] + (['CAG'] if cag_m else [])
colors = ['#3498db', '#e74c3c'][:len(sys_names)]

# Response Time
times = [rag_m['time']] + ([cag_m['time']] if cag_m else [])
ax[0].bar(sys_names, times, color=colors); ax[0].set_title('⚡ Response Time (s)')
for i,v in enumerate(times): ax[0].text(i, v, f'{v:.3f}', ha='center', va='bottom', fontweight='bold')

# BERTScore
berts = [rag_m['bert']] + ([cag_m['bert']] if cag_m else [])
ax[1].bar(sys_names, berts, color=colors); ax[1].set_title('🎯 BERTScore F1'); ax[1].set_ylim(0,1)
for i,v in enumerate(berts): ax[1].text(i, v, f'{v:.4f}', ha='center', va='bottom', fontweight='bold')

# Quality Metrics
x = np.arange(4); w = 0.35
rv = [rag_m['comp'], rag_m['hall'], rag_m['recall'], rag_m['eir']]
ax[2].bar(x - w/2, rv, w, label='RAG', color='#3498db')
if cag_m:
    cv = [cag_m['comp'], cag_m['hall'], cag_m['recall'], cag_m['eir']]
    ax[2].bar(x + w/2, cv, w, label='CAG', color='#e74c3c')
ax[2].set_xticks(x); ax[2].set_xticklabels(['Comp', 'Hall', 'Recall', 'EIR'])
ax[2].set_title('📊 Quality'); ax[2].legend(); ax[2].set_ylim(0,1)

plt.tight_layout()
os.makedirs('../logs', exist_ok=True)
plt.savefig('../logs/eval_results.png', dpi=150)
plt.show()
print("✅ Saved: logs/eval_results.png")

## 📝 Summary & Export
Simpan hasil ke CSV & JSON, tampilkan key findings dari evaluasi.

In [ ]:
# 💾 SAVE & SUMMARY
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
df.to_csv(f'../logs/eval_{ts}.csv', index=False)
json.dump({'ts': ts, 'rag': rag_m, 'cag': cag_m}, open(f'../logs/eval_{ts}.json', 'w'), indent=2)

print("\n" + "="*60)
print("🎯 KEY FINDINGS")
print("="*60)
print(f"RAG: {rag_m['time']:.3f}s | BERTScore: {rag_m['bert']:.4f} | Completeness: {rag_m['comp']:.4f}")
if cag_m:
    print(f"CAG: {cag_m['time']:.3f}s | BERTScore: {cag_m['bert']:.4f} | Cache: {cag_m['cache']*100:.0f}%")
    print(f"\n💡 CAG is {speedup:.1f}x faster with comparable quality!")
print("\n✅ EVALUATION COMPLETE!")